# Attention From Scratch
### Why Transformers are powerful, and why they are expensive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zelihaguven/llm-concepts/blob/main/notebooks/architecture/01_transformer_attention.ipynb)

Companion notebook for the first post of the **Model Architecture** series.

What we do here:
1. Build self-attention by hand on a tiny example and watch the shapes flow.
2. Check that our version matches PyTorch's built-in implementation.
3. Add a causal mask (how GPT-style models avoid seeing the future).
4. Split attention into multiple heads.
5. Look inside a real pretrained model (GPT-2 small).
6. Measure how the cost of attention grows with sequence length.

Everything runs on a laptop CPU. GPT-2 small is ~500 MB to download the first time.

## 0. Setup

In [ ]:
import sys
import shutil
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/zelihaguven/llm-concepts.git"
IN_COLAB = "google.colab" in sys.modules

def find_root(start):
    """Return the folder that contains src/attention.py, searching inside `start`."""
    matches = sorted(Path(start).glob("**/src/attention.py"), key=lambda p: len(p.parts))
    return matches[0].parent.parent if matches else None

if IN_COLAB:
    # On Colab, clone the repo so we can import the shared code in src/.
    clone_dir = Path("/content/llm-concepts")
    ROOT = find_root(clone_dir) if clone_dir.exists() else None
    if ROOT is None:
        shutil.rmtree(clone_dir, ignore_errors=True)      # remove a broken earlier clone
        result = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)],
                                capture_output=True, text=True)
        if result.returncode != 0:
            raise RuntimeError("Could not clone the repo. Is it pushed and public?\n" + result.stderr)
        ROOT = find_root(clone_dir)
else:
    # Locally, walk up from the notebook folder until we find the repo root.
    ROOT = Path.cwd().resolve()
    while not (ROOT / "src" / "attention.py").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "attention.py").exists():
        ROOT = None

if ROOT is None:
    raise FileNotFoundError(
        "src/attention.py not found. Check that the repo on GitHub contains src/attention.py "
        "(not only the notebook), then run this cell again."
    )

sys.path.insert(0, str(ROOT))
FIG_DIR = ROOT / "figures" / "01_attention"
FIG_DIR.mkdir(parents=True, exist_ok=True)
print("Repo root:", ROOT)
print("Figures will be saved to:", FIG_DIR)

In [ ]:
# Run the setup cell above first: it puts the repo root on the Python path.
import time

import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

from src.attention import scaled_dot_product_attention, causal_mask, MultiHeadAttention

torch.manual_seed(0)
np.random.seed(0)
print("PyTorch", torch.__version__)

In [ ]:
def plot_attention(weights, query_labels, key_labels=None, ax=None, title=None, annotate=None):
    """Heatmap of an attention matrix: rows = queries (who is looking),
    columns = keys (who is being looked at)."""
    key_labels = key_labels or query_labels
    w = weights.detach().cpu().numpy()
    if ax is None:
        _, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(w, cmap="viridis", vmin=0)
    ax.set_xticks(range(len(key_labels)))
    ax.set_xticklabels(key_labels, rotation=90)
    ax.set_yticks(range(len(query_labels)))
    ax.set_yticklabels(query_labels)
    ax.set_xlabel("key (being looked at)")
    ax.set_ylabel("query (looking)")
    if title:
        ax.set_title(title)
    # Write the numbers inside the cells for small matrices.
    if annotate is None:
        annotate = w.shape[0] <= 8
    if annotate:
        for i in range(w.shape[0]):
            for j in range(w.shape[1]):
                ax.text(j, i, f"{w[i, j]:.2f}", ha="center", va="center",
                        color="white" if w[i, j] < w.max() * 0.6 else "black", fontsize=8)
    return im

## 1. A toy example: watching the shapes

Five tokens, each represented by a random 8-dimensional vector. In a real model these vectors would come from an embedding layer; here random numbers are enough to see how attention moves data around.

Each token vector is projected three times:
- **Query (Q):** what this token is looking for
- **Key (K):** the label this token offers to others
- **Value (V):** the content this token hands over if someone attends to it

In [ ]:
tokens = ["the", "cat", "sat", "on", "mat"]
seq_len, d_model = len(tokens), 8

x = torch.randn(1, seq_len, d_model)          # (batch, seq, d_model)

W_q = nn.Linear(d_model, d_model, bias=False)
W_k = nn.Linear(d_model, d_model, bias=False)
W_v = nn.Linear(d_model, d_model, bias=False)

with torch.no_grad():
    q, k, v = W_q(x), W_k(x), W_v(x)
    scores = q @ k.transpose(-2, -1)

print("input x      :", tuple(x.shape), " -> (batch, tokens, features)")
print("queries q    :", tuple(q.shape))
print("keys k       :", tuple(k.shape))
print("values v     :", tuple(v.shape))
print("scores q@k^T :", tuple(scores.shape), " -> one score for every (query, key) pair")

## 2. Our own attention

Now run the full recipe from `src/attention.py`: scores, scale by √d, softmax, mix the values. Every row of the weight matrix should sum to 1: each token splits 100% of its attention across the tokens it can see.

In [ ]:
with torch.no_grad():
    out, weights = scaled_dot_product_attention(q, k, v)

print("output shape :", tuple(out.shape))
print("weights shape:", tuple(weights.shape))
print("\nattention weights (rounded):")
print(weights[0].round(decimals=2))
print("\nrow sums:", weights[0].sum(dim=-1).round(decimals=4))
assert torch.allclose(weights.sum(dim=-1), torch.ones(1, seq_len)), "rows must sum to 1"

fig, ax = plt.subplots(figsize=(4.5, 4))
plot_attention(weights[0], tokens, ax=ax, title="Toy attention (random weights)")
plt.tight_layout()
fig.savefig(FIG_DIR / "toy_attention.png", dpi=150)
plt.show()

### Why divide by √d?

Dot products of random vectors grow with their dimension. Without scaling, larger models would produce huge scores, and softmax would put almost all the weight on a single token. Here we measure the average **largest** attention weight over random queries and 10 random keys, with and without scaling.

In [ ]:
n_keys, trials = 10, 500
print(f"{'d':>5} | {'max weight, unscaled':>22} | {'max weight, scaled':>20}")
print("-" * 54)
for d in [8, 64, 512]:
    qs = torch.randn(trials, 1, d)
    ks = torch.randn(trials, n_keys, d)
    raw = qs @ ks.transpose(-2, -1)                  # (trials, 1, n_keys)
    unscaled = torch.softmax(raw, dim=-1).max(dim=-1).values.mean().item()
    scaled = torch.softmax(raw / d ** 0.5, dim=-1).max(dim=-1).values.mean().item()
    print(f"{d:>5} | {unscaled:>22.3f} | {scaled:>20.3f}")
print(f"\n(uniform attention over {n_keys} keys would give {1 / n_keys:.3f})")

## 3. Sanity check against PyTorch

If our from-scratch version is right, it should match `torch.nn.functional.scaled_dot_product_attention` up to tiny floating-point differences.

In [ ]:
with torch.no_grad():
    reference = F.scaled_dot_product_attention(q, k, v)

max_diff = (out - reference).abs().max().item()
print(f"max absolute difference: {max_diff:.2e}")
assert torch.allclose(out, reference, atol=1e-6)
print("✓ Our attention matches PyTorch's built-in implementation.")

## 4. Causal mask: no peeking at the future

A model like GPT generates text left to right, so token *i* may only look at tokens 0 … *i*. We enforce this with a lower-triangular mask: hidden positions get a score of −∞, which softmax turns into exactly 0.

In [ ]:
mask = causal_mask(seq_len)
print("mask (1 = allowed to look):")
print(mask.int())

with torch.no_grad():
    out_causal, weights_causal = scaled_dot_product_attention(q, k, v, mask=mask)
    reference_causal = F.scaled_dot_product_attention(q, k, v, is_causal=True)

assert torch.allclose(out_causal, reference_causal, atol=1e-6)
print("\n✓ Causal version also matches PyTorch (is_causal=True).")

fig, axes = plt.subplots(1, 2, figsize=(9, 4))
plot_attention(weights[0], tokens, ax=axes[0], title="Full attention")
plot_attention(weights_causal[0], tokens, ax=axes[1], title="Causal attention")
plt.tight_layout()
fig.savefig(FIG_DIR / "full_vs_causal.png", dpi=150)
plt.show()

Notice the first token: under the causal mask it can only see itself, so its row is a single 1.00.

## 5. Multi-head attention

Instead of one attention over all 8 features, we split them into 2 heads of 4 features each. Each head has its own view of the tokens and produces its own weight matrix. The results are then joined back together.

We also check our `MultiHeadAttention` against PyTorch's `nn.MultiheadAttention` by copying the same weights into both.

In [ ]:
torch.manual_seed(0)
n_heads = 2
mha = MultiHeadAttention(d_model=d_model, n_heads=n_heads)

with torch.no_grad():
    out_mha, weights_mha = mha(x)

print("output shape :", tuple(out_mha.shape), " -> same as the input")
print("weights shape:", tuple(weights_mha.shape), " -> (batch, heads, query, key)")

# Copy our weights into PyTorch's module and compare.
ref_mha = nn.MultiheadAttention(embed_dim=d_model, num_heads=n_heads, bias=False, batch_first=True)
with torch.no_grad():
    ref_mha.in_proj_weight.copy_(torch.cat([mha.q_proj.weight, mha.k_proj.weight, mha.v_proj.weight]))
    ref_mha.out_proj.weight.copy_(mha.out_proj.weight)
    ref_out, ref_weights = ref_mha(x, x, x, need_weights=True, average_attn_weights=False)

assert torch.allclose(out_mha, ref_out, atol=1e-6)
assert torch.allclose(weights_mha, ref_weights, atol=1e-6)
print("✓ MultiHeadAttention matches nn.MultiheadAttention.")

fig, axes = plt.subplots(1, n_heads, figsize=(4.5 * n_heads, 4))
for h in range(n_heads):
    plot_attention(weights_mha[0, h], tokens, ax=axes[h], title=f"Head {h}")
plt.tight_layout()
fig.savefig(FIG_DIR / "multi_head_toy.png", dpi=150)
plt.show()

The two heads see the same tokens but distribute their attention differently. In a trained model, this is what lets different heads specialise in different relationships.

## 6. Looking inside a real model: GPT-2 small

Heatmaps from random weights don't mean anything, because nothing has been learned. So we use our own code to **understand** attention, and a pretrained model to **observe** it.

GPT-2 small has 12 layers × 12 heads = 144 attention heads. We give it the classic sentence and ask: when the model reads **"it"**, which heads look back at **"animal"**?

Note: `attn_implementation="eager"` is needed because the faster default implementation does not return attention weights.

In [ ]:
from transformers import AutoModel, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("gpt2")
try:
    gpt2 = AutoModel.from_pretrained("gpt2", attn_implementation="eager")
except TypeError:  # older transformers versions
    gpt2 = AutoModel.from_pretrained("gpt2")
gpt2.eval()

sentence = "The animal didn't cross the street because it was too tired."
inputs = tokenizer(sentence, return_tensors="pt")

with torch.no_grad():
    outputs = gpt2(**inputs, output_attentions=True)

# outputs.attentions: one tensor per layer, each (batch, heads, seq, seq)
attn = torch.stack(outputs.attentions).squeeze(1)      # (layers, heads, seq, seq)
labels = [tokenizer.decode([t]).strip() for t in inputs["input_ids"][0]]

n_layers, n_heads_gpt2 = attn.shape[0], attn.shape[1]
print(f"{n_layers} layers x {n_heads_gpt2} heads, {len(labels)} tokens")
print(list(enumerate(labels)))

In [ ]:
it_idx = labels.index("it")
animal_idx = labels.index("animal")
street_idx = labels.index("street")

# How much does "it" attend to "animal" (and to "street") in every head?
it_to_animal = attn[:, :, it_idx, animal_idx]          # (layers, heads)
it_to_street = attn[:, :, it_idx, street_idx]

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(it_to_animal.numpy(), cmap="viridis", vmin=0)
ax.set_xlabel("head")
ax.set_ylabel("layer")
ax.set_xticks(range(n_heads_gpt2))
ax.set_yticks(range(n_layers))
ax.set_title('Attention from "it" to "animal", every head of GPT-2 small')
fig.colorbar(im, ax=ax, label="attention weight")
plt.tight_layout()
fig.savefig(FIG_DIR / "gpt2_it_to_animal_grid.png", dpi=150)
plt.show()

top = torch.topk(it_to_animal.flatten(), k=5)
print('Top 5 heads where "it" looks at "animal":')
for value, flat_idx in zip(top.values, top.indices):
    layer, head = divmod(flat_idx.item(), n_heads_gpt2)
    print(f"  layer {layer:>2}, head {head:>2}: animal = {value:.3f} | street = {it_to_street[layer, head]:.3f}")

In [ ]:
# Full heatmaps for the 3 strongest heads.
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))
for ax, flat_idx in zip(axes, top.indices[:3]):
    layer, head = divmod(flat_idx.item(), n_heads_gpt2)
    plot_attention(attn[layer, head], labels, ax=ax, title=f"GPT-2 layer {layer}, head {head}", annotate=False)
    ax.axhline(it_idx, color="red", linewidth=0.8, alpha=0.6)   # the "it" row
plt.tight_layout()
fig.savefig(FIG_DIR / "gpt2_top_heads.png", dpi=150)
plt.show()

# Where does "it" look in the single strongest head?
best_layer, best_head = divmod(top.indices[0].item(), n_heads_gpt2)
row = attn[best_layer, best_head, it_idx, : it_idx + 1]
fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(len(row)), row.numpy())
ax.set_xticks(range(len(row)))
ax.set_xticklabels(labels[: it_idx + 1], rotation=45)
ax.set_ylabel("attention weight")
ax.set_title(f'What "it" attends to (layer {best_layer}, head {best_head})')
plt.tight_layout()
fig.savefig(FIG_DIR / "gpt2_it_bar.png", dpi=150)
plt.show()

### Something you will probably notice: the first token

Many heads put a large share of their attention on the **first token** ("The"), whatever the query is. This is a known pattern often called an *attention sink*: since every row must sum to 1, a head that has nothing useful to look at needs somewhere to "park" its attention, and the first token (visible to every position under the causal mask) becomes that default.

The cell below measures how common this is.

In [ ]:
# Average attention paid to token 0, over all queries after the first one.
first_token_share = attn[:, :, 1:, 0].mean(dim=-1)        # (layers, heads)
share = first_token_share.flatten()
print(f"Heads where the average attention on the first token is > 50%: "
      f"{(share > 0.5).sum().item()} of {share.numel()}")

fig, ax = plt.subplots(figsize=(7, 5.5))
im = ax.imshow(first_token_share.numpy(), cmap="magma", vmin=0, vmax=1)
ax.set_xlabel("head")
ax.set_ylabel("layer")
ax.set_xticks(range(n_heads_gpt2))
ax.set_yticks(range(n_layers))
ax.set_title("Average attention on the first token")
fig.colorbar(im, ax=ax)
plt.tight_layout()
fig.savefig(FIG_DIR / "gpt2_first_token_sink.png", dpi=150)
plt.show()

**Write down what you actually see before drafting the post.** Some heads may show a clear "it → animal" link, others may not. Reporting that honestly ("only a few of 144 heads do this, and many heads mostly look at the first token") is more interesting and more credible than claiming every head is meaningful.

## 7. The quadratic cost

Attention compares **every** token with **every** other token. For *n* tokens that is an *n × n* matrix, so doubling the sequence length makes the matrix four times larger.

We time a single attention head (d = 64) for increasing sequence lengths and compute the memory one attention matrix needs in float32.

In [ ]:
def time_attention(seq_len, d=64, repeats=5):
    q = torch.randn(1, seq_len, d)
    k = torch.randn(1, seq_len, d)
    v = torch.randn(1, seq_len, d)
    scaled_dot_product_attention(q, k, v)        # warm-up run, not timed
    times = []
    for _ in range(repeats):
        start = time.perf_counter()
        scaled_dot_product_attention(q, k, v)
        times.append(time.perf_counter() - start)
    return float(np.median(times))

lengths = [128, 256, 512, 1024, 2048, 4096]
with torch.no_grad():
    times_s = [time_attention(n) for n in lengths]
matrix_mb = [n * n * 4 / 1e6 for n in lengths]       # float32 = 4 bytes

print(f"{'tokens':>7} | {'time (ms)':>10} | {'one matrix (MB)':>16}")
print("-" * 40)
for n, t, m in zip(lengths, times_s, matrix_mb):
    print(f"{n:>7} | {t * 1000:>10.2f} | {m:>16.1f}")

# Slope on a log-log plot: ~2 means quadratic growth.
# The smallest sizes are dominated by fixed overhead, so fit on the larger ones.
slope_time = np.polyfit(np.log(lengths[2:]), np.log(times_s[2:]), 1)[0]
slope_mem = np.polyfit(np.log(lengths), np.log(matrix_mb), 1)[0]
print(f"\nlog-log slope, time  : {slope_time:.2f}")
print(f"log-log slope, memory: {slope_mem:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
ref = np.array(lengths, dtype=float)

axes[0].loglog(lengths, np.array(times_s) * 1000, "o-", label="measured")
axes[0].loglog(ref, (times_s[-1] * 1000) * (ref / ref[-1]) ** 2, "--", alpha=0.6, label="n² reference")
axes[0].set_title(f"Attention time (slope ≈ {slope_time:.2f})")
axes[0].set_xlabel("sequence length (tokens)")
axes[0].set_ylabel("time (ms)")
axes[0].legend()

axes[1].loglog(lengths, matrix_mb, "o-", color="tab:orange", label="one attention matrix")
axes[1].set_title(f"Attention matrix memory (slope ≈ {slope_mem:.2f})")
axes[1].set_xlabel("sequence length (tokens)")
axes[1].set_ylabel("memory (MB, float32)")
axes[1].legend()

for ax in axes:
    ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
fig.savefig(FIG_DIR / "quadratic_cost.png", dpi=150)
plt.show()

### Scaling it up to a whole model

One matrix is only part of the story: every head in every layer has its own. If a naive implementation kept all of them in memory at once (as plain training does, to run backpropagation), GPT-2 small would need:

In [ ]:
layers, heads = 12, 12   # GPT-2 small
for n in [1024, 4096, 16384]:
    total_gb = n * n * 4 * layers * heads / 1e9
    print(f"{n:>6} tokens: {total_gb:8.2f} GB of attention matrices (batch size 1)")
print("\n(GPT-2 small itself supports at most 1024 tokens; the larger rows are hypothetical.)")

Techniques like FlashAttention avoid storing these matrices by computing attention in small blocks, which solves most of the **memory** problem. The amount of **computation**, however, still grows quadratically with sequence length.

## 8. Summary

*Fill this in after running the notebook, using your own numbers. These three findings become sections 7 and 8 of the post.*

1. **Attention is a short recipe:** scores → scale → mask → softmax → mix values. Our from-scratch version matches PyTorch exactly (single head, causal, and multi-head).
2. **Inside GPT-2:** _which heads linked "it" to "animal", how strong was the link, and how many heads mostly attended to the first token?_
3. **The cost:** _measured slope ≈ …; at 4096 tokens one matrix is ≈ 67 MB, and a whole naive GPT-2-sized model would need ≈ … GB._

**Next post:** while generating text, a model stores the keys and values of all previous tokens (the KV-cache), and that memory also grows with context length. We will look at how **latent attention** shrinks it.